# Part 3 — Reinforcement Learning (Q-Learning)

**Environment**: Grid World — agent navigates from start to goal while avoiding obstacles  
**Algorithm**: Q-Learning (off-policy TD control)  
**Hyperparameters explored**: alpha (learning rate), gamma (discount factor), epsilon (exploration rate)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Libraries loaded.')

## 1. Grid World Environment

```
S . . . . . . . . .
. # # # . . . . . .
. . . # . # # # . .
. . . # . # . . . .
. . . . . # . # # .
. . . . . . . . . G
```
- S = Start (top-left)
- G = Goal (bottom-right)
- # = Obstacle
- . = Free cell


In [ ]:
class GridWorld:
    """
    Configurable grid world environment for Q-Learning experiments.
    Actions: 0=Up, 1=Down, 2=Left, 3=Right
    Rewards: -1 per step, -5 for hitting wall/obstacle, +100 for goal
    """
    ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    N_ACTIONS = 4

    def __init__(self, grid_size=10, obstacles=None, start=(0, 0), goal=None):
        self.rows = grid_size
        self.cols = grid_size
        self.start = start
        self.goal = goal if goal else (grid_size - 1, grid_size - 1)
        self.obstacles = set(obstacles) if obstacles else self._default_obstacles()
        self.state = self.start

    def _default_obstacles(self):
        return {
            (1,1),(1,2),(1,3),(2,3),(2,5),(2,6),(2,7),
            (3,3),(3,5),(4,5),(4,7),(4,8),
        }

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        dr, dc = self.ACTIONS[action]
        nr, nc = self.state[0] + dr, self.state[1] + dc
        if not (0 <= nr < self.rows and 0 <= nc < self.cols):
            return self.state, -1, False
        if (nr, nc) in self.obstacles:
            return self.state, -5, False
        self.state = (nr, nc)
        if self.state == self.goal:
            return self.state, 100, True
        return self.state, -1, False

    def render(self, path=None):
        grid = np.zeros((self.rows, self.cols))
        for (r, c) in self.obstacles:
            grid[r, c] = -1
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(grid, cmap='gray_r', vmin=-1, vmax=1)
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) == self.start:
                    ax.text(c, r, 'S', ha='center', va='center', fontsize=14, color='blue', fontweight='bold')
                elif (r, c) == self.goal:
                    ax.text(c, r, 'G', ha='center', va='center', fontsize=14, color='green', fontweight='bold')
                elif (r, c) in self.obstacles:
                    ax.text(c, r, '#', ha='center', va='center', fontsize=12, color='white')
        if path:
            path_r = [p[0] for p in path]
            path_c = [p[1] for p in path]
            ax.plot(path_c, path_r, 'b-o', markersize=6, linewidth=2, alpha=0.8, label='Agent path')
            ax.legend(fontsize=12)
        ax.set_xticks(range(self.cols))
        ax.set_yticks(range(self.rows))
        ax.grid(True, color='gray', linewidth=0.5)
        ax.set_title('Grid World Environment', fontsize=14)
        plt.tight_layout()
        plt.show()

env = GridWorld()
env.render()
print(f'Grid: {env.rows}x{env.cols} | Obstacles: {len(env.obstacles)} | Start: {env.start} | Goal: {env.goal}')

## 2. Q-Learning Agent

Q-Learning update rule:

Q(s,a) = Q(s,a) + alpha * [r + gamma * max_a Q(s',a') - Q(s,a)]

Exploration strategy: epsilon-greedy with exponential decay.


In [ ]:
class QLearningAgent:
    def __init__(self, n_rows, n_cols, n_actions,
                 alpha=0.1, gamma=0.95, epsilon=1.0,
                 epsilon_min=0.01, epsilon_decay=0.995):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.q_table = np.zeros((n_rows, n_cols, n_actions))

    def choose_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.q_table[state[0], state[1]]))

    def update(self, state, action, reward, next_state, done):
        current_q = self.q_table[state[0], state[1], action]
        target = reward if done else reward + self.gamma * np.max(self.q_table[next_state[0], next_state[1]])
        self.q_table[state[0], state[1], action] += self.alpha * (target - current_q)

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def get_path(self, env, max_steps=200):
        state = env.reset()
        path = [state]
        for _ in range(max_steps):
            action = int(np.argmax(self.q_table[state[0], state[1]]))
            next_state, _, done = env.step(action)
            path.append(next_state)
            state = next_state
            if done:
                break
        return path

print('QLearningAgent defined.')

## 3. Training


In [ ]:
def train_agent(env, agent, n_episodes=1000, max_steps=300):
    rewards, lengths, success_rate = [], [], []
    recent = []
    for ep in range(n_episodes):
        state = env.reset()
        total_r, steps, done = 0, 0, False
        while not done and steps < max_steps:
            action = agent.choose_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            state = next_state
            total_r += reward
            steps += 1
        agent.decay_epsilon()
        rewards.append(total_r)
        lengths.append(steps)
        recent.append(1 if done and state == env.goal else 0)
        if len(recent) > 50:
            recent.pop(0)
        success_rate.append(np.mean(recent))
    return np.array(rewards), np.array(lengths), np.array(success_rate)


env = GridWorld()
agent = QLearningAgent(env.rows, env.cols, env.N_ACTIONS,
                       alpha=0.1, gamma=0.95, epsilon=1.0)
rewards, lengths, success = train_agent(env, agent, n_episodes=1500)
print(f'Final success rate (last 50 eps): {success[-1]*100:.1f}%')
print(f'Best episode reward: {rewards.max():.1f}')

In [ ]:
def smooth(arr, w=50):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(smooth(rewards), color='steelblue')
axes[0].set_title('Episode Reward (smoothed)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')

axes[1].plot(smooth(lengths), color='coral')
axes[1].set_title('Episode Length (smoothed)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Steps')

axes[2].plot(success, color='green')
axes[2].set_title('Success Rate (last 50 eps)')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Success Rate')
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize learned path
env_vis = GridWorld()
path = agent.get_path(env_vis)
env_vis.render(path=path)
print(f'Path length: {len(path)-1} steps | Reached goal: {path[-1] == env.goal}')

In [ ]:
# Q-table heatmap: max Q-value per cell
max_q = np.max(agent.q_table, axis=2)

plt.figure(figsize=(8, 7))
sns.heatmap(max_q, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Q-Table: Max Q-Value per Cell')
plt.xlabel('Column')
plt.ylabel('Row')
plt.tight_layout()
plt.show()

## 4. Hyperparameter Experiments


In [ ]:
# Experiment 1: Learning rate (alpha)
alphas = [0.01, 0.1, 0.3, 0.7]
fig, ax = plt.subplots(figsize=(10, 5))
for alpha in alphas:
    e = GridWorld()
    a = QLearningAgent(e.rows, e.cols, e.N_ACTIONS, alpha=alpha, gamma=0.95)
    _, _, sr = train_agent(e, a, n_episodes=1000)
    ax.plot(sr, label=f'alpha={alpha}')
ax.set_title('Effect of Learning Rate (alpha) on Success Rate')
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Experiment 2: Discount factor (gamma)
gammas = [0.5, 0.75, 0.9, 0.99]
fig, ax = plt.subplots(figsize=(10, 5))
for gamma in gammas:
    e = GridWorld()
    a = QLearningAgent(e.rows, e.cols, e.N_ACTIONS, alpha=0.1, gamma=gamma)
    _, _, sr = train_agent(e, a, n_episodes=1000)
    ax.plot(sr, label=f'gamma={gamma}')
ax.set_title('Effect of Discount Factor (gamma) on Success Rate')
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Experiment 3: Epsilon decay rate
decays = [0.990, 0.995, 0.999]
fig, ax = plt.subplots(figsize=(10, 5))
for decay in decays:
    e = GridWorld()
    a = QLearningAgent(e.rows, e.cols, e.N_ACTIONS, alpha=0.1, gamma=0.95, epsilon_decay=decay)
    _, _, sr = train_agent(e, a, n_episodes=1500)
    ax.plot(sr, label=f'epsilon_decay={decay}')
ax.set_title('Effect of Epsilon Decay on Success Rate')
ax.set_xlabel('Episode')
ax.set_ylabel('Success Rate')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Modified Environment — Dense Obstacles


In [ ]:
dense_obstacles = {
    (1,1),(1,2),(1,3),(1,4),(2,4),(2,6),(2,7),
    (3,0),(3,4),(3,6),(4,2),(4,4),(4,6),(4,8),
    (5,2),(5,8),(6,2),(6,4),(6,6),(6,8),(7,4),(7,6),(8,0),(8,4),
}
env_hard = GridWorld(obstacles=dense_obstacles)
agent_hard = QLearningAgent(env_hard.rows, env_hard.cols, env_hard.N_ACTIONS,
                             alpha=0.1, gamma=0.95, epsilon=1.0, epsilon_decay=0.997)
r_h, l_h, sr_h = train_agent(env_hard, agent_hard, n_episodes=2000)
path_hard = agent_hard.get_path(GridWorld(obstacles=dense_obstacles))
env_hard.render(path=path_hard)
print(f'Dense grid — final success rate: {sr_h[-1]*100:.1f}% | Path length: {len(path_hard)-1}')

## 6. Analysis and Reflection

### Hyperparameter Insights

| Hyperparameter | Low value | High value | Optimal range |
|----------------|-----------|------------|---------------|
| **alpha (learning rate)** | Slow convergence | Unstable Q-values | 0.05 - 0.2 |
| **gamma (discount)** | Myopic / greedy | Values distant future | 0.9 - 0.99 |
| **epsilon-decay** | Exploits too early | Too slow to converge | 0.99 - 0.997 |

### Key Observations
1. Learning curves: rewards go from very negative (random exploration) toward positive as Q-values converge.
2. Q-table heatmap: cells near goal have high values; obstacle-adjacent cells have learned low values.
3. Dense obstacles require more episodes and slower epsilon decay to avoid local optima.
4. High gamma (0.99) is critical for long-horizon navigation — the agent must plan ahead.

### Real-World Scaling Challenges
- Tabular Q-Learning cannot handle continuous state spaces (e.g. real sensor readings)
- Solution: Deep Q-Networks (DQN) use neural networks to approximate the Q-function
- Industrial robots cannot afford millions of random exploration steps — simulation-first is essential

## 7. Ethical and Practical Considerations

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Reward hacking** | Agent finds shortcuts that maximize reward but violate intent | Careful reward design; constraint-based RL |
| **Safety during exploration** | Random actions can cause physical damage in real systems | Safe exploration; simulation-first; action constraints |
| **Non-stationarity** | Real environment changes over time | Periodic retraining; online learning with drift detection |
| **Interpretability** | Q-table decisions hard to explain to operators | Visualize learned policy; add explanation layer |
